# 03 — Open-answer judge (plan.md §A9, G9)

Scores saved open-question responses with a local LLM (`medprune/judge.py`): one forward pass per unique
(question, reference, response), verdict = P(Yes) / (P(Yes) + P(No)). No model, method or image is shown to the judge.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **on** · input dataset `medprune-judge-items`
(`items.jsonl`, written on the laptop by `python -m medprune.judge export --results results/all`).
**Output:** `judge/verdicts.jsonl` → merge into `results/all/judge/verdicts.jsonl` on the laptop (append; keys are unique).

In [ ]:
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
BATCH_SIZE = 16
LIMIT = None  # e.g. 64 for a smoke test

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:  # the judge model is ungated; the token only avoids Hugging Face rate limits
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("cvpr")
    print("HF token loaded")
except Exception as error:
    print("secret 'cvpr' not attached; continuing without a token:", type(error).__name__)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/cvpr && git clone --depth 1 https://github.com/pushkar1887/cvpr /tmp/cvpr
%cd /tmp/cvpr
!git log --oneline -1
!bash scripts/setup_env.sh

In [ ]:
import glob
from pathlib import Path

ITEMS = Path(sorted(glob.glob("/kaggle/input/**/items.jsonl", recursive=True))[0])
OUT = Path("/kaggle/working/judge/verdicts.jsonl")
print(ITEMS, sum(1 for _ in ITEMS.open(encoding="utf-8")), "items")
LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
!python -m medprune.judge run --items {ITEMS} --out {OUT} --model {JUDGE_MODEL} --batch-size {BATCH_SIZE} {LIMIT_ARG}

In [ ]:
import json
import numpy as np

v = [json.loads(line) for line in OUT.open(encoding="utf-8")]
p = np.array([x["p_correct"] for x in v])
mass = np.array([x["verdict_mass"] for x in v])
print(f"{len(v)} verdicts | P(correct) >= 0.5: {np.mean(p >= 0.5):.3f} | undecided (0.2-0.8): {np.mean((p > 0.2) & (p < 0.8)):.3f}")
print(f"verdict mass: median {np.median(mass):.3f}, share < 0.9: {np.mean(mass < 0.9):.3f}  (low mass = judge not answering Yes/No)")